In [1]:
from checker import run
from pyspark.sql import SparkSession
import subprocess
import os
import json
import pandas as pd
from datetime import datetime

In [2]:
def call_ssh_command(command: str, host: str = "gateway.st"):
    return subprocess.run(
        ["ssh", host, command], 
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
    ).stdout

In [ ]:
# Проверка лаб запускается локально на моей машине, но ничего 
# не мешает использовать и удалённый спарк-кластер.
spark = (
    SparkSession
    .builder
    .appName("test")
    .master("local[*]")
    .config("spark.driver.memory", "30g")
    .config("spark.executor.memory", "30g")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .getOrCreate()
)

In [5]:
# Путь до приватного датасета, на котором проверяем задачи
DATA_PATH = "/dev/shm/bigdata20/"

In [6]:
# Просто набор переменных, чтобы различать группы студентов и номер попытки
# Ни на что не влияет, можно просто забить на это.
CHECK_ATTEMPT_NUM = 2
STUDENTS_TERM = f"2024-Spring-RU"
STUDENTS_LIST_DIR = f"../../students/{STUDENTS_TERM}"
STUDENTS_NOTEBOOKS = f"../../students/{STUDENTS_TERM}/spark-lab/attempt-{CHECK_ATTEMPT_NUM}"

In [7]:
# Подгружаем список студентов, чтобы знать чьи лабы нужно будет проверять
students = pd.read_csv(f"{STUDENTS_LIST_DIR}/all_groups_data_uid.csv")

In [ ]:
# Читаем датасеты в кэш перед запуском проверок

itmo_posts = spark.read.json(os.path.join(DATA_PATH, "posts_api.json")).cache()
followers_posts_likes = spark.read.parquet(os.path.join(DATA_PATH, "followers_posts_likes.parquet")).cache()
followers_posts = spark.read.json(os.path.join(DATA_PATH, "followers_posts_api_final.json")).cache()

itmo_posts.write.format("noop").mode("overwrite").save()
followers_posts_likes.write.format("noop").mode("overwrite").save()
followers_posts.write.format("noop").mode("overwrite").save()

with open("../../course-material/labs/spark-lab/emojis_sentiment.json", "r") as f:
    emojis_data = json.loads(f.read())

In [11]:
RESULTS = dict()

In [9]:
def check_notebook(path: str):
    result = run(
        nb_path=f"{path}",
        spark=spark,
        itmo_posts=itmo_posts,
        followers_posts_likes=followers_posts_likes,
        followers_posts=followers_posts,
        emojis_data=emojis_data,
        # skip_tasks=("task_4",)
    )
    return result

In [10]:
def check_all_notebooks(
    df: pd.DataFrame
):
    for _, row in df.iterrows():
        login = row["login"]
        print(f"{datetime.utcnow()} Checking {login}...")
        if login not in RESULTS.keys():
            RESULTS[login] = check_notebook(
                path=f"{STUDENTS_NOTEBOOKS}/{login}-{CHECK_ATTEMPT_NUM}.ipynb"
            )

In [12]:
students = students.sort_values(by=["login"], inplace=False).reset_index(drop=True)

In [13]:
students

,login,password,group,full_name,uid
0,acherkasskaja-412638,855LaS7D6o,2024-Spring-RU,Cherkasskaja Anna Vitalevna,10362
1,aemeljanov-411284,X81ZJu8m00,2024-Spring-RU,Emeljanov Andrej Mihajlovich,10304
2,agudkov-411193,78T78h4uYh,2024-Spring-RU,Gudkov Aleksandr Sergeevich,10301
3,ahurmatov-412611,xF94511xag,2024-Spring-RU,Hurmatov Artem Maratovich,10361
4,akalachin-411433,x41N9vy2b7,2024-Spring-RU,Kalachin Artem Nikolaevich,10309
...,...,...,...,...,...
75,vselivanova-412278,7F83GB21BQ,2024-Spring-RU,Selivanova Viktorija Valerevna,10345
76,vtihevich-285560,g61pr42S0e,2024-Spring-RU,Tihevich Valerija Vitalevna,10353
77,vvetrov-414193,3IhHz7162j,2024-Spring-RU,Vetrov Vladislav Olegovich,10295
78,vvinnichenko-411068,0W47g3Xf5W,2024-Spring-RU,Vinnichenko Veronika Sergeevna,10296


In [15]:
%%time
# Запустить проверку всех лаб
check_all_notebooks(students)

2024-06-14 17:30:28.648496 Checking acherkasskaja-412638...
WebUI: http://lvm:4040
Checking task task_1a...
Checking task task_1b...
Checking task task_1c...
Checking task task_2a...
Checking task task_2b...


Checking task task_3...
Checking task task_4...


Checking task task_5...


Checking task task_6...


2024-06-14 17:32:31.592910 Checking aemeljanov-411284...
WebUI: http://lvm:4040
Checking task task_1a...
Checking task task_1b...
Checking task task_1c...
Checking task task_2a...
Checking task task_2b...


...

2024-06-14 19:33:02.199576 Checking vvinnichenko-411068...
WebUI: http://lvm:4040
Checking task task_1a...
Checking task task_1b...
Checking task task_1c...
Checking task task_2a...
Checking task task_2b...


Checking task task_3...
Checking task task_4...


Checking task task_5...


Checking task task_6...


2024-06-14 19:36:06.979972 Checking zscheglov-412747...
CPU times: user 29.5 s, sys: 23 s, total: 52.5 s
Wall time: 2h 5min 38s


In [17]:
with open(f"tmp-{STUDENTS_TERM}-{CHECK_ATTEMPT_NUM}.json", "w") as f:
    f.write(json.dumps(RESULTS))

In [12]:
%%time
# Запустить проверку только одного определённого ноутбука
_result = check_notebook(
    path=f"/mnt/hgfs/Projects/StudentsCluster/st_k8s/students/2024-Spring-EN/spark-lab/attempt-1/gourab-dey-1.ipynb"
)

WARN [task_5]: Unexpected args signature. Expected: {'df': 'pyspark.sql.dataframe.DataFrame', 'F': 'pyspark.sql.functions', 'W': 'pyspark.sql.window.Window', 'top_n_likers': 'int'}. Got: {'df': 'DataFrame', 'F': 'pyspark.sql.functions', 'W': 'pyspark.sql.window.Window', 'top_n_likers': 'int'}
Func name: task_5
CPU times: user 268 ms, sys: 12.8 ms, total: 281 ms
Wall time: 286 ms


In [13]:
_result

{'task_1a': {'is_completed': False, 'reason': None},
 'task_1b': {'is_completed': False, 'reason': None},
 'task_1c': {'is_completed': False, 'reason': None},
 'task_2a': {'is_completed': False, 'reason': None},
 'task_2b': {'is_completed': False, 'reason': None},
 'task_3': {'is_completed': False, 'reason': None},
 'task_4': {'is_completed': False, 'reason': None},
 'task_5': {'is_completed': False,
  'reason': "Unexpected args signature. Expected: {'df': 'pyspark.sql.dataframe.DataFrame', 'F': 'pyspark.sql.functions', 'W': 'pyspark.sql.window.Window', 'top_n_likers': 'int'}. Got: {'df': 'DataFrame', 'F': 'pyspark.sql.functions', 'W': 'pyspark.sql.window.Window', 'top_n_likers': 'int'}\nFunc name: task_5"},
 'task_6': {'is_completed': False, 'reason': None}}